# Terms & Conditions Analyzer
This notebook extracts the core logic of the Streamlit application for interactive use.

In [12]:
!pip install streamlit langchain langchain-openai langchain-community python-dotenv beautifulsoup4


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 56.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 83.4 MB/s eta 0:00:00:00:010:01


In [5]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document

# Load environment variables
load_dotenv()

/tmp/ipykernel_2402/14099088.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


False

In [6]:
# Initialize the Language Model
llm = ChatOpenAI(
    model="qwen/qwen3-coder-next",  # or any OpenRouter model
    openai_api_base="https://openrouter.ai/api/v1",
    max_tokens=1000,
)

In [ ]:
# Document Loading
# You can change this to either a URL or raw text
input_method = "URL" # Change to "TEXT" to use raw text

if input_method == "URL":
    url = "https://developer.chrome.com/docs/webstore/program-policies/terms" # Example URL
    loader = WebBaseLoader(url)
    documents = loader.load()
    text = documents[0].page_content
    print(f"Loaded {len(text)} characters from URL.")
else:
    text = """Paste your Terms and Conditions text here."""
    print(f"Loaded {len(text)} characters from text.")

















Google Chrome Web Store Developer Agreement  |  Chrome Web Store - Program Policies  |  Chrome for Developers






      
      Skip to main content
    























    Docs
  
    





Build with Chrome
Learn how Chrome works, participate in origin trials, and build with Chrome everywhere.




                      Web Platform
                    





                      Capabilities
                    





                      ChromeDriver
                    





                      Extensions
                    





                      Chrome Web Store
                    





                      Chromium
                    





                      Web on Android
                    





                      Origin trials
                    





                      Release notes
                    






Productivity
Create the best experience for your users with the web's best tools.



                      DevTools
      

In [8]:
# Prompts and Chains Setup
parser = StrOutputParser()

prompt1 = PromptTemplate(
    template="Summarize the following set of terms and conditions in a clear, very brief and consise way: {text}",
    input_variables=["text"]
)

prompt2 = PromptTemplate(
    template="Based on the following set of terms and conditions, list out the most offensive ones as bullet points very briefly: {text} \n ",
    input_variables=["text"]
)

summary_chain = prompt1 | llm | parser
offensive_chain = prompt2 | llm | parser

In [9]:
# Execution
print("--- SUMMARY ---")
summary = summary_chain.invoke({"text": text})
print(summary)

print("\n--- OFFENSIVE TERMS ---")
offensive_terms = offensive_chain.invoke({"text": text})
if not offensive_terms or len(offensive_terms.strip()) <= 5:
    offensive_terms = "No highly offensive terms found."
print(offensive_terms)

--- SUMMARY ---
**Brief Summary of Google Chrome Web Store Developer Agreement:**

Developers must accept the Agreement and pay a registration fee to publish extensions/apps on the Chrome Web Store. Key points:

- **Obligations:** Comply with the Agreement, Program Policies, and applicable laws. Ensure user privacy, security, and accurate disclosures. Provide timely support (3 business days for paid products; 24h for urgent issues).
- **Content Rules:** No illegal, malicious, deceptive, infringing, or harmful content. No NPAPI unless approved.
- **Licensing:** You grant Google a license to distribute your product; users receive a license to use it. You retain ownership and must have rights to grant the license.
- **Reviews & Takedowns:** Google may review, reject, or remove noncompliant products. You may remove yours, but previously downloaded copies remain usable by users (unless removal is due to legal/IP issues).
- **Payments:** Google handles no payments or taxes for free or paid p

In [10]:
# Interactive Chat Interface (Run this cell to chat)
chat_history = [summary, offensive_terms]

print("Chat with the Terms & Conditions (type 'exit' to quit):")
while True:
    user_input = input("You: ")
    if user_input.lower() in ['exit', 'quit']:
        break
    
    chat_history.append(user_input + " briefly without any jargon")
    
    response = llm.invoke(chat_history)
    print(f"\nAssistant: {response.content}\n")
    
    chat_history.append(response.content)


Chat with the Terms & Conditions (type 'exit' to quit):

Assistant: Here’s a straightforward breakdown of the most problematic parts of the Chrome Web Store Developer Agreement from a developer’s point of view:

- **Google can remove your app anytime, for any reason—even just “we think it’s risky”—with no real way to appeal.**  
- **Google won’t be held responsible if something goes wrong** (e.g., your app breaks, you lose data, or Google removes it unfairly).  
- **You have to protect Google from lawsuits**—even if the problem is your app, a bug, copyright claims, taxes, or even if Google messed up.  
- **You give Google broad, permanent rights to use your app (and even your source code)** for推广, support, and more—even after you stop using the Store.  
- **You’re fully responsible for *everything* under your developer account**, including apps you didn’t write or if your account is hacked.  
- **Google decides how visible your app is, your ratings, and can hide or remove it based on u

KeyboardInterrupt: Interrupted by user

In [13]:
import streamlit as st
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate 
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document
from dotenv import load_dotenv

load_dotenv()
# api_key = st.secrets["OPENAI_API_KEY"]

st.set_page_config(page_title="T&C Analyzer", page_icon="⚖️", layout="wide")

st.title("⚖️ Terms & Conditions Analyzer")
st.write("Paste the URL of the Terms & Conditions or paste the text directly.")

# setup LLM & Chains
@st.cache_resource
def get_model():
    llm = ChatOpenAI(
        model="qwen/qwen3-coder-next",  # or any OpenRouter model
        openai_api_base="https://openrouter.ai/api/v1",
        max_tokens=1000, # limits the output length
    )
    return llm

try:
    model = get_model()
except Exception as e:
    st.error(f"Error loading model: {e}")
    st.stop()

# initialize session state variables
if "summary" not in st.session_state:
    st.session_state.summary = None
if "offensive_terms" not in st.session_state:
    st.session_state.offensive_terms = None
if "messages" not in st.session_state:
    st.session_state.messages = []
if "chat_history" not in st.session_state:
    st.session_state.chat_history = []

# file uploader & input options
input_method = st.radio("Choose input method:", ["URL Link", "Paste Text"])

documents = []
text_loaded = False


if input_method == "URL Link":
    url_input = st.text_input("Enter the URL of the Terms & Conditions:")
    if url_input:
        try:
            with st.spinner("Scraping webpage..."):
                loader = WebBaseLoader(url_input)
                documents = loader.load()
                text_loaded = True
        except Exception as e:
            st.error(f"Failed to load URL: {e}")

elif input_method == "Paste Text":
    raw_text = st.text_area("Paste the Terms & Conditions text here:", height=200)
    if raw_text.strip():
        documents = [Document(page_content=raw_text)]
        text_loaded = True

button=st.button("Analyze Document")

if text_loaded and st.session_state.summary is None:
    if button:
        with st.spinner("Analyzing document..."):
            
            if not documents:
                st.error("Document is empty or could not be parsed.")
                st.stop()
                
            whole_text = documents[0].page_content
            
            if len(whole_text.strip()) == 0:
                st.error("The loaded text is empty.")
                st.stop()
            
            parser = StrOutputParser()
            prompt1 = PromptTemplate(
                template="Summarize the following set of terms and conditions in a clear, very brief and consise way: {text}",
                input_variables=["text"]
            )
            prompt2 = PromptTemplate(
                template="Based on the following set of terms and conditions, list out the most offensive ones as bullet points very briefly: {text} \n ",
                input_variables=["text"],
                MAX_TOKENS_PER_REQUEST=200
            )
            
            chain1 = prompt1 | model | parser
            chain2 = prompt2 | model | parser
            
            status_text = st.empty()
            status_text.text("Summarizing the document...")
            
            # summarize the whole text
            final_summary = chain1.invoke({"text": whole_text})
            
            status_text.text("Extracting offensive terms from the document...")
            
            # find offensive terms from the whole text
            final_offensive = chain2.invoke({"text": whole_text})
            
            if not final_offensive or len(final_offensive.strip()) <= 5:
                 final_offensive = "No highly offensive terms found."
            
            st.session_state.summary = final_summary
            st.session_state.offensive_terms = final_offensive
            
            status_text.empty()
            
            # initialize display messages 
            st.session_state.messages = []
            
            st.session_state.chat_history = [final_summary, final_offensive]
            st.rerun()

if st.session_state.summary is not None:
    st.markdown("Analysis Results:")
    col1, col2 = st.columns(2)
    with col1:
        st.subheader("📝 Summary:")
        st.info(st.session_state.summary)
    with col2:
        st.subheader("🚩 Offensive Terms:")
        st.warning(st.session_state.offensive_terms)
        
    st.divider()
    st.markdown("💬 Chat:")
    
    # display chat messages from history on app rerun
    for message in st.session_state.messages:
        with st.chat_message(message["role"]):
            st.markdown(message["content"])

    # react to user input
    if prompt := st.chat_input("Ask a question about the Terms & Conditions..."):
        # display user message in chat message container
        st.chat_message("user").markdown(prompt)
        
        # add user message to UI and model chat history
        st.session_state.messages.append({"role": "user", "content": prompt})
        st.session_state.chat_history.append(prompt + "briefly without any jargon")

        with st.chat_message("assistant"):
            with st.spinner("Thinking..."):
                chatresult = model.invoke(st.session_state.chat_history)
                response_text = chatresult.content
                st.markdown(response_text)
                
        # add AI response to UI and model chat history
        st.session_state.messages.append({"role": "assistant", "content": response_text})
        st.session_state.chat_history.append(response_text)


2026-07-10 15:55:29.296 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-10 15:55:29.297 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-10 15:55:29.480 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-07-10 15:55:29.481 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-10 15:55:29.482 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-10 15:55:29.485 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-10 15:55:29.486 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn